In [19]:
# HyperOpt 를 찾기 위한 기본 루틴 정리본 , 
# 본 파일을 save_as 이니셜_1.모델축약어_FindHO.ipynb 로 저장하세요~ ex) lkj_1.XGB_FindHO.ipynb

In [20]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [21]:
import os

import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

# module_path = os.path.abspath(os.path.join('..'))
# if module_path not in sys.path:
#     sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics         import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics         import roc_auc_score
from sklearn.metrics         import precision_recall_curve, classification_report
from sklearn.datasets         import make_classification

# Model import
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.ensemble       import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.linear_model   import LinearRegression
from sklearn.svm            import SVC
from sklearn.metrics        import classification_report
from xgboost                import XGBClassifier
from xgboost                import plot_importance
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier

# hyperopt 용
from hyperopt               import hp


# 사용자 Functions import
import HyperParams          as HP 
import utils.data_sampling  as ds 

from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

In [22]:
# 결과 받을 딕셔너리
results = {}

In [23]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [24]:
# 2. Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')

In [25]:
# 3. 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [26]:
# 4.1 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

In [27]:
# 5.1 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [29]:
#6 하이퍼파라미터
tuner = uu.HyperOptTuner(max_evals=100, random_state=23)

# 모델별 스페이스 생성 : LightGBM
lgbm_search_space = {
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'num_leaves': hp.quniform('num_leaves', 31, 256, 1),
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    'feature_fraction': hp.uniform('feature_fraction', 0.5, 1.0),
    'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1.0),
    'bagging_freq': hp.quniform('bagging_freq', 0, 5, 1),
    'min_data_in_leaf': hp.quniform('min_data_in_leaf', 20, 200, 5),
    'lambda_l1': hp.uniform('lambda_l1', 0.0, 5.0),
    'lambda_l2': hp.uniform('lambda_l2', 0.0, 5.0),
    # 불균형 데이터 대응
    'scale_pos_weight': hp.choice('scale_pos_weight', [1, int(len(y_train)/sum(y_train))])
}

# 모델 생성 (LightGBM)
lgbm = LGBMClassifier()

# 파라미터 지정
best_params, best_lgbm, trials, exec_time = tuner.tune(
    lgbm, X_tr, y_tr, X_val, y_val, lgbm_search_space
)

# best모델로 결과출력
model_name = 'lgb_ho_best'
results[model_name] = uu.get_model_train_eval(
    best_lgbm, model_name, X_train, X_test, y_train, y_test, best_params
)


LGBMClassifier 튜닝 시작
  0%|          | 0/100 [00:00<?, ?trial/s, best loss=?]

[LightGBM] [Warning] min_data_in_leaf is set=160.0, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=160.0
[LightGBM] [Warning] feature_fraction is set=0.5965749484346121, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5965749484346121
[LightGBM] [Warning] lambda_l1 is set=1.6420868771198043, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.6420868771198043
[LightGBM] [Warning] lambda_l2 is set=0.7277782946697364, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.7277782946697364
[LightGBM] [Warning] bagging_fraction is set=0.5576796500555039, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5576796500555039
[LightGBM] [Warning] bagging_freq is set=3.0, subsample_freq=0 will be ignored. Current value: bagging_freq=3.0
Error in objective function: Parameter min_data_in_leaf should be of type int, got "160.0"
[LightGBM] [Warning] min_data_in_leaf is set=80.0, min_child_samples=20 will be ignored. Current valu

KeyError: 'scores'

In [ ]:
# 7 시각화
mo.model_metrics_graph(results, 'cb모델 성능지표 비교')